In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.neighbors import NearestNeighbors

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
mcdir = f'{projdir}/snmCT/mc'

In [ ]:
nuclei = pd.read_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t', index_col=0)
early = nuclei[nuclei['early_ipsc']]
nuclei.shape, early.shape

In [ ]:
adata = sc.read_h5ad(f'{mcdir}/h5ad/integrate_m3c/merged.processed.harmony.h5ad')
adata.obs.index = [x.rsplit('-', 1)[0] for x in adata.obs.index]
ips_adata = adata[adata.obs.index.isin(early.index)].copy()
ips_adata.obs['ips_subtype'] = early['ips_subtype'].reindex(ips_adata.obs.index)
ips_adata.shape

### methylation PCs exactly as in Fig 6I/J (R01d), computed once on all early-iPSC LE/SE nuclei and then held fixed

In [ ]:
sc.pp.highly_variable_genes(ips_adata, n_top_genes=20000)
sc.tl.pca(ips_adata, n_comps=50)
pcs = ips_adata.obsm['X_pca']
obs = ips_adata.obs[['line', 'assay', 'ips_subtype']].copy()
obs['ips_subtype'] = obs['ips_subtype'].map({'IPS_main': 'iPSC-LE', 'IPS_alt': 'iPSC-SE'})
pd.crosstab([obs['assay'], obs['line']], obs['ips_subtype'])

In [ ]:
def same_donor_frac(X, lines, k=50):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, ind = nn.kneighbors(X)
    ind = ind[:, 1:]
    return (lines[ind] == lines[:, None]).mean(axis=1)

### reference: full early-iPSC set, same kNN rule (50 neighbours, self excluded)

In [ ]:
lines = obs['line'].values
full = obs.copy()
full['same_donor'] = same_donor_frac(pcs, lines)
full_sum = full.groupby(['line', 'ips_subtype'])['same_donor'].mean().unstack()
full_sum['LE_minus_SE'] = full_sum['iPSC-LE'] - full_sum['iPSC-SE']
full_sum

### balanced design: per assay, the smallest donor x subtype group size

In [ ]:
n_assay = obs.groupby(['assay', 'line', 'ips_subtype']).size().groupby('assay').min()
n_assay

In [ ]:
groups = {(a, l, s): np.where((obs['assay'] == a) & (obs['line'] == l) & (obs['ips_subtype'] == s))[0]
          for a in n_assay.index for l in donors for s in ['iPSC-LE', 'iPSC-SE']}
rows = []
for seed in range(500):
    rng = np.random.default_rng(seed)
    idx = np.concatenate([rng.choice(v, n_assay[k[0]], replace=False) for k, v in groups.items()])
    frac = same_donor_frac(pcs[idx], lines[idx])
    tmp = obs.iloc[idx].copy()
    tmp['same_donor'] = frac
    summ = tmp.groupby(['line', 'ips_subtype'])['same_donor'].mean().reset_index()
    summ['seed'] = seed
    rows.append(summ)
bal = pd.concat(rows, ignore_index=True)
len(idx), bal.shape

In [ ]:
bal_wide = bal.pivot_table(index=['seed', 'line'], columns='ips_subtype', values='same_donor').reset_index()
bal_wide['LE_minus_SE'] = bal_wide['iPSC-LE'] - bal_wide['iPSC-SE']
bal_sum = bal_wide.groupby('line').apply(lambda x: pd.Series({
    'LE_mean': x['iPSC-LE'].mean(), 'SE_mean': x['iPSC-SE'].mean(),
    'diff_mean': x['LE_minus_SE'].mean(), 'diff_median': x['LE_minus_SE'].median(),
    'diff_q025': x['LE_minus_SE'].quantile(0.025), 'diff_q975': x['LE_minus_SE'].quantile(0.975),
    'frac_LE_gt_SE': (x['LE_minus_SE'] > 0).mean()}))
bal_sum

### chance level in the balanced sample = 1/4

In [ ]:
bal.groupby('ips_subtype')['same_donor'].mean()

In [ ]:
bal_wide.to_csv(f'{outdir}/ips_subtype_nn_prop_balanced_500.csv', sep='\t', index=False)
bal_sum.to_csv(f'{outdir}/ips_subtype_nn_prop_balanced_500_summary.csv', sep='\t')
full_sum.to_csv(f'{outdir}/ips_subtype_nn_prop_full_exact_knn.csv', sep='\t')

In [ ]:
!date